# E5: fair tuning of XLM-R large and IndicBERT v2 for emotion

Both models get the same budget: up to 8 epochs with early stopping, mixed precision, and three learning rates each. The learning rate is chosen on **validation only** (seed 13), then each model runs seeds 13, 21 and 42 on test once. The plan was committed before any run (commit `3abde63`).

**How to run**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Runtime > **Run all**. Allow Google Drive access when asked.
3. Keep the tab open. Expect about 2.5 to 3.5 hours (12 training runs).
4. The last cell downloads `e5_results.zip`. Send that zip back.

If Colab disconnects or the GPU quota runs out, reconnect later and **Run all** again: finished runs are kept on Drive and skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
assert torch.cuda.get_device_properties(0).total_memory > 14e9, 'Need a 15 GB or larger GPU (T4 or better)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/mhrisk_e5'
os.makedirs(SAVE, exist_ok=True)
os.environ['MHRISK_RUNS_DIR'] = '/content/runs'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

In [ ]:
COMMIT = '3abde636fe8638ebe2bc4edc1dc5a59456026679'  # the pre-registered E5 plan
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/faizdevx/Multilingual-Mental-Health-Risk-.git /content/repo
%cd /content/repo
!git fetch -q origin && git checkout -q {COMMIT} && git log --oneline -1
!pip install -q -r requirements-colab.txt 2>&1 | tail -n 2

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py > /dev/null
!python scripts/freeze_splits.py --check

In [ ]:
# all 12 runs; progress lines appear as each run finishes
!python scripts/e5_tune.py --device cuda --save {SAVE} 2>&1 | grep --line-buffered -E 'stage|chosen|seed [0-9]+:|Error|Traceback|error'

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{SAVE}/E5_[34]_*/selection.json')):
    s = json.load(open(f)); print(s['grid'], '-> chosen LR', s['chosen_encoder_lr'], [(c['encoder_lr'], round(c['val_macro_f1'], 4)) for c in s['candidates']])
for exp in ('E5_3_xlmr_large_emotion_tuned', 'E5_4_indicbertv2_emotion_tuned'):
    for seed in (13, 21, 42):
        m = json.load(open(f'{SAVE}/{exp}/seed_{seed}/metrics.json'))
        t = m['test']['emotion']
        print(exp, seed, 'epochs', m['epochs_run'], '| test', round(t['macro_f1'], 4), {k: round(v['macro_f1'], 4) for k, v in t['by_language'].items()})

## Download the results
Metrics and per-example probabilities only; no text, no model weights.

In [ ]:
!cd {SAVE} && rm -f e5_results.zip && zip -qr e5_results.zip E5_* -x '*/resume*' && unzip -l e5_results.zip | tail -n 1
from google.colab import files
files.download(f'{SAVE}/e5_results.zip')